# Fatigue, couverture des calendriers et résultats en Ligue des Champions

**Question :** l'équipe qui a eu le plus de repos avant un match de C1 gagne-t-elle plus souvent, une fois le niveau des deux équipes pris en compte ?

L'historique vient de `games.csv`; les analyses portent uniquement sur les matchs `CL`. Les indicateurs de repos ne sont interprétés que lorsque `games.csv` contient au moins un match de championnat (`domestic_league`) pour le club durant la saison. Cela signale une couverture absente, sans garantir que tout le calendrier de championnat est complet. Les indicateurs sont au niveau du club et ne mesurent pas les minutes jouées par les joueurs.

## Définitions et précautions

- **Jours de repos :** jours calendaires entre le match de C1 et le match précédent du club, toutes compétitions visibles dans `games.csv` confondues.
- **Matchs sur 30 jours :** matchs datés entre les 30 jours avant le match de C1 inclus et la veille du match. Le match étudié n'est pas compté.
- Un match du même jour n'est pas considéré comme antérieur, car l'heure et l'ordre des rencontres ne sont pas garantis.
- **Compétitions par saison :** nombre de `competition_id` distincts joués pendant la saison entière; c'est descriptif, pas une variable connue en début de saison.
- **Couverture du championnat :** indicateur vrai si au moins un match `domestic_league` apparaît pour ce club et cette saison. S'il n'y en a aucun, le calendrier est incomplet dans le fichier et le club-match est exclu des analyses de repos. Une présence ne prouve toutefois pas que tous les matchs de championnat sont recensés.
- L'Elo utilisé est la dernière note publiée avant le match. Les matchs nuls valent 0,5 point dans la comparaison Elo. L'analyse de repos utilise les matchs où les repos diffèrent et où les deux équipes ont une couverture de championnat détectée.

In [ ]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLEU, ORANGE, VERT, GRIS = '#2a78d6', '#eb6834', '#1baf7a', '#b5b3ad'
ROOT = Path.cwd()
if not (ROOT / 'data' / 'games.csv').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
GAMES_PATH = DATA_DIR / 'games.csv'
EVENTS_PATH = DATA_DIR / 'game_events.csv'
ELO_PATH = DATA_DIR / 'EloRatings.csv'
OUTPUT_DIR = ROOT / 'output'
if not GAMES_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable : {GAMES_PATH}. Place games.csv dans data/.')
if not ELO_PATH.exists():
    raise FileNotFoundError(f'Fichier introuvable : {ELO_PATH}. Place EloRatings.csv dans data/.')

games = pd.read_csv(GAMES_PATH)
elo = pd.read_csv(ELO_PATH, parse_dates=['date'])
required = {
    'game_id', 'date', 'season', 'competition_id', 'competition_type',
    'home_club_id', 'away_club_id', 'home_club_name', 'away_club_name',
    'home_club_goals', 'away_club_goals',
}
missing = required - set(games.columns)
if missing:
    raise ValueError(f'Colonnes manquantes dans games.csv : {sorted(missing)}')
games['date'] = pd.to_datetime(games['date'], errors='coerce').dt.normalize()
games = games.dropna(subset=['date', 'home_club_id', 'away_club_id', 'competition_id']).copy()
games['competition_id'] = games['competition_id'].astype(str)
games['competition_type'] = games['competition_type'].fillna('inconnu').astype(str)

home = games[['game_id', 'date', 'season', 'competition_id', 'competition_type', 'home_club_id', 'away_club_id', 'home_club_name', 'away_club_name', 'home_club_goals', 'away_club_goals']].copy()
home.columns = ['game_id', 'date', 'season', 'competition_id', 'competition_type', 'club_id', 'opponent_id', 'club', 'adversaire', 'buts_pour', 'buts_contre']
home['cote'] = 'Domicile'
away = games[['game_id', 'date', 'season', 'competition_id', 'competition_type', 'away_club_id', 'home_club_id', 'away_club_name', 'home_club_name', 'away_club_goals', 'home_club_goals']].copy()
away.columns = ['game_id', 'date', 'season', 'competition_id', 'competition_type', 'club_id', 'opponent_id', 'club', 'adversaire', 'buts_pour', 'buts_contre']
away['cote'] = 'Extérieur'
club_games = pd.concat([home, away], ignore_index=True)
club_games['resultat'] = np.select(
    [club_games['buts_pour'] > club_games['buts_contre'], club_games['buts_pour'] < club_games['buts_contre']],
    ['Victoire', 'Défaite'], default='Nul'
)
print(f"{len(games):,} matchs toutes compétitions; {games['competition_id'].nunique():,} compétitions")
print(f"{elo['club'].nunique():,} clubs avec Elo; dates de {elo['date'].min().date()} à {elo['date'].max().date()}")

## Corriger les scores de tirs au but

Quand `game_events.csv` est disponible, les scores des matchs de C1 identifiés avec une séance (`Shootout`) sont remplacés par le compte des buts (`Goals`) avant les tirs au but. Sinon, les scores de `games.csv` sont conservés et le notebook le signale.

In [ ]:
ucl_ids = set(games.loc[games['competition_id'].eq('CL'), 'game_id'])
shootout_ids = set()
if EVENTS_PATH.exists():
    events = pd.read_csv(EVENTS_PATH, usecols=['game_id', 'type', 'club_id'])
    event_type = events['type'].astype(str).str.strip().str.lower()
    shootout_ids = set(events.loc[event_type.eq('shootout'), 'game_id']) & ucl_ids
    goals = events.loc[event_type.eq('goals')].groupby(['game_id', 'club_id']).size()
    correct = club_games['game_id'].isin(shootout_ids)
    club_games.loc[correct, 'buts_pour'] = [
        goals.get((game_id, club_id), 0)
        for game_id, club_id in zip(club_games.loc[correct, 'game_id'], club_games.loc[correct, 'club_id'])
    ]
    club_games.loc[correct, 'buts_contre'] = [
        goals.get((game_id, opponent_id), 0)
        for game_id, opponent_id in zip(club_games.loc[correct, 'game_id'], club_games.loc[correct, 'opponent_id'])
    ]
    club_games.loc[correct, 'resultat'] = np.select(
        [club_games.loc[correct, 'buts_pour'] > club_games.loc[correct, 'buts_contre'], club_games.loc[correct, 'buts_pour'] < club_games.loc[correct, 'buts_contre']],
        ['Victoire', 'Défaite'], default='Nul'
    )
    print(f'{len(shootout_ids)} match(s) de C1 avec tirs au but corrigé(s) depuis les événements.')
else:
    print('game_events.csv absent : scores enregistrés utilisés; vérifier les séances de tirs au but.')

ucl_teams = club_games.loc[club_games['competition_id'].eq('CL')].copy()
print(f"{ucl_teams['game_id'].nunique():,} matchs de C1 retenus.")

## Indicateurs avant chaque match de C1

Les matchs sont triés par club et par date. La recherche s'arrête au début du jour du match de C1, donc le match étudié et les rencontres du même jour sont exclus.

In [ ]:
competitions_saison = (club_games.groupby(['club_id', 'season'])['competition_id']
                       .nunique().rename('nb_competitions_saison'))
matchs_ligue_saison = (
    club_games.loc[club_games['competition_type'].eq('domestic_league')]
    .groupby(['club_id', 'season']).size()
    .rename('matchs_championnat_saison')
)
historiques = {club: np.sort(group['date'].to_numpy(dtype='datetime64[ns]'))
              for club, group in club_games.groupby('club_id')}
repos, nb_30j = [], []
for row in ucl_teams.itertuples():
    dates = historiques[row.club_id]
    match_date = np.datetime64(row.date, 'ns')
    end = np.searchsorted(dates, match_date, side='left')
    repos.append(np.nan if end == 0 else int((match_date - dates[end - 1]) / np.timedelta64(1, 'D')))
    start = np.searchsorted(dates, match_date - np.timedelta64(30, 'D'), side='left')
    nb_30j.append(int(end - start))
ucl_teams['jours_repos'] = repos
ucl_teams['matchs_30j_avant'] = nb_30j
ucl_teams = ucl_teams.merge(competitions_saison, left_on=['club_id', 'season'], right_index=True, how='left')
ucl_teams = ucl_teams.merge(matchs_ligue_saison, left_on=['club_id', 'season'], right_index=True, how='left')
ucl_teams['matchs_championnat_saison'] = ucl_teams['matchs_championnat_saison'].fillna(0).astype(int)
ucl_teams['couverture_championnat_detectee'] = ucl_teams['matchs_championnat_saison'].gt(0)

columns = [
    'game_id', 'date', 'season', 'club_id', 'club', 'opponent_id', 'adversaire', 'cote',
    'buts_pour', 'buts_contre', 'resultat', 'jours_repos', 'matchs_30j_avant',
    'nb_competitions_saison', 'matchs_championnat_saison', 'couverture_championnat_detectee',
]
fatigue = ucl_teams[columns].sort_values(['date', 'game_id', 'club_id']).reset_index(drop=True)
print(f"Clubs-matchs sans match de championnat détecté dans leur saison : {(~fatigue['couverture_championnat_detectee']).sum():,}")
fatigue.head()

## Niveau avant le match : Elo

On relie les noms de clubs à `EloRatings.csv`, puis on prend pour chaque club le dernier Elo strictement antérieur à la date du match. Le score attendu est la valeur Elo classique : il dépend de l'écart de niveau. On compare ensuite les points réellement pris par le club le plus reposé à ses points attendus. Les clubs sans correspondance Elo restent identifiés comme non évaluables sur ce volet.

In [ ]:
def normaliser_nom(nom):
    decomposed = unicodedata.normalize('NFKD', str(nom))
    sans_accents = ''.join(caractere for caractere in decomposed if not unicodedata.combining(caractere))
    return ''.join(caractere for caractere in sans_accents.lower() if caractere.isalnum())

correspondance_elo = {
    'Arsenal FC': 'Arsenal', 'Aston Villa': 'Aston Villa', 'Chelsea FC': 'Chelsea', 'Leicester City': 'Leicester',
    'Liverpool FC': 'Liverpool', 'Manchester City': 'Man City', 'Manchester United': 'Man United',
    'Newcastle United': 'Newcastle', 'Tottenham Hotspur': 'Tottenham',
    'Athletic Bilbao': 'Ath Bilbao', 'Atlético de Madrid': 'Ath Madrid', 'FC Barcelona': 'Barcelona',
    'Girona FC': 'Girona', 'Málaga CF': 'Malaga', 'Real Madrid': 'Real Madrid', 'Real Sociedad': 'Sociedad',
    'Sevilla FC': 'Sevilla', 'Valencia CF': 'Valencia', 'Villarreal CF': 'Villarreal',
    '1.FC Union Berlin': 'Union Berlin', 'Bayer 04 Leverkusen': 'Leverkusen', 'Bayern Munich': 'Bayern Munich',
    'Borussia Dortmund': 'Dortmund', 'Borussia Mönchengladbach': 'MGladbach', 'Eintracht Frankfurt': 'Ein Frankfurt',
    'FC Schalke 04': 'Schalke 04', 'RB Leipzig': 'RB Leipzig', 'TSG 1899 Hoffenheim': 'Hoffenheim',
    'VfB Stuttgart': 'Stuttgart', 'VfL Wolfsburg': 'Wolfsburg',
    'AC Milan': 'Milan', 'Associazione Sportiva Roma': 'Roma', 'Atalanta BC': 'Atalanta',
    'Bologna Football Club 1909': 'Bologna', 'Inter Milan': 'Inter', 'Juventus FC': 'Juventus',
    'SSC Napoli': 'Napoli', 'Società Sportiva Lazio S.p.A.': 'Lazio',
    'AS Monaco': 'Monaco', 'LOSC Lille': 'Lille', 'Montpellier HSC': 'Montpellier', 'Olympique Lyon': 'Lyon',
    'Olympique Marseille': 'Marseille', 'Paris Saint-Germain': 'Paris SG', 'RC Lens': 'Lens',
    'Stade Brestois 29': 'Brest', 'Stade Rennais FC': 'Rennes',
    'Ajax Amsterdam': 'Ajax', 'Feyenoord Rotterdam': 'Feyenoord', 'PSV Eindhoven': 'PSV Eindhoven',
    'FC Porto': 'Porto', 'SC Braga': 'Sp Braga', 'SL Benfica': 'Benfica', 'Sporting CP': 'Sp Lisbon',
    'Club Brugge KV': 'Club Brugge', 'KAA Gent': 'Gent', 'KRC Genk': 'Genk', 'RSC Anderlecht': 'Anderlecht',
    'Royal Antwerp FC': 'Antwerp', 'Union Saint-Gilloise': 'St. Gilloise',
    'Austria Vienna': 'Austria Wien', 'Fußballclub Red Bull Salzburg': 'Salzburg',
    'Sportklub Puntigamer Sturm Graz': 'Sturm Graz', 'Celtic FC': 'Celtic', 'Rangers FC': 'Rangers',
    'FC Copenhagen': 'FC Kobenhavn', 'FC Midtjylland': 'Midtjylland', 'FC Nordsjaelland': 'Nordsjaelland',
    'Athlitiki Enosi Konstantinoupoleos': 'AEK', 'Olympiakos Syndesmos Filathlon Peiraios': 'Olympiakos',
    'Basaksehir FK': 'Buyuksehyr', 'Beşiktaş Jimnastik Kulübü': 'Besiktas', 'Galatasaray': 'Galatasaray',
    'AO FK Zenit Sankt-Peterburg': 'Zenit', 'FK Krasnodar': 'FC Krasnodar', 'FK Rostov': 'Rostov',
    'FK Spartak Moskva': 'Spartak Moskva', 'PFK CSKA Moskva': 'CSKA Moskva',
    'Футбольный клуб "Локомотив" Москва': 'Lok Moskva',
    'CFR Cluj': 'CFR Cluj', 'FCSB': 'Steaua', 'Legia Warszawa': 'Legia',
    'Fotballklubben Bodø/Glimt': 'Bodoe Glimt', 'Malmö Fotbollförening': 'Malmoe',
}
elo_noms = {normaliser_nom(nom): nom for nom in elo['club'].dropna().unique()}
correspondance_valide = {
    normaliser_nom(club): elo_noms[normaliser_nom(nom_elo)]
    for club, nom_elo in correspondance_elo.items()
    if normaliser_nom(nom_elo) in elo_noms
}
fatigue['elo_club'] = fatigue['club'].map(lambda nom: correspondance_valide.get(normaliser_nom(nom)))
elo_tries = elo.assign(elo_club=elo['club'].map(lambda nom: elo_noms[normaliser_nom(nom)]))
elo_tries = elo_tries.sort_values('date')[['date', 'elo_club', 'elo']].rename(columns={'date': 'date_elo'})
fatigue = pd.merge_asof(
    fatigue.sort_values('date'),
    elo_tries,
    left_on='date', right_on='date_elo', by='elo_club',
    direction='backward', allow_exact_matches=False,
).sort_values(['date', 'game_id', 'club_id']).reset_index(drop=True)
fatigue = fatigue.rename(columns={'elo': 'elo_avant_match'})
fatigue['points_reels'] = fatigue['resultat'].map({'Victoire': 1.0, 'Nul': 0.5, 'Défaite': 0.0})

elo_matchs = fatigue.pivot(index='game_id', columns='cote', values='elo_avant_match')
elo_matchs.columns = [f'elo_{cote.lower()}' for cote in elo_matchs.columns]
elo_matchs['points_attendus_domicile'] = 1 / (1 + 10 ** ((elo_matchs['elo_extérieur'] - elo_matchs['elo_domicile']) / 400))
elo_matchs['points_attendus_extérieur'] = 1 - elo_matchs['points_attendus_domicile']
fatigue = fatigue.merge(elo_matchs[['points_attendus_domicile', 'points_attendus_extérieur']], on='game_id', how='left')
fatigue['points_attendus_elo'] = np.where(
    fatigue['cote'].eq('Domicile'), fatigue['points_attendus_domicile'], fatigue['points_attendus_extérieur']
)
fatigue['ecart_points_attendus_elo'] = fatigue['points_reels'] - fatigue['points_attendus_elo']
print(f"Matchs avec Elo pré-match pour les deux clubs : {elo_matchs.dropna(subset=['elo_domicile', 'elo_extérieur']).shape[0]:,} / {len(elo_matchs):,}")
print(f"Clubs de C1 sans correspondance Elo : {sorted(fatigue.loc[fatigue['elo_club'].isna(), 'club'].unique())}")

## Livrable CSV

Une ligne par club et par match de C1; chaque match apparaît donc deux fois, une fois par équipe.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUTPUT_DIR / 'fatigue_par_match.csv'
fatigue.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
print(f"CSV créé : {CSV_PATH} ({len(fatigue):,} lignes, {fatigue['game_id'].nunique():,} matchs)")
print(f"Jours de repos manquants : {fatigue['jours_repos'].isna().sum():,}")

## Le repos apporte-t-il quelque chose au-delà de l'Elo ?

On compare le club le plus reposé à son score attendu selon l'écart d'Elo pré-match. La comparaison ne garde que les matchs où les deux clubs ont une couverture de championnat détectée, les deux Elo sont disponibles, et les repos diffèrent. L'écart moyen aux points attendus est descriptif; son intervalle de confiance aide à voir si les données distinguent cet écart de zéro, sans prouver l'absence d'un effet.

In [ ]:
measures = [
    'jours_repos', 'resultat', 'elo_avant_match', 'points_reels',
    'points_attendus_elo', 'couverture_championnat_detectee',
]
pairs = fatigue.pivot(index='game_id', columns='cote', values=measures)
pairs.columns = [f'{measure}_{side.lower()}' for measure, side in pairs.columns]
rest_difference = pairs['jours_repos_domicile'] - pairs['jours_repos_extérieur']
valid = (
    pairs['couverture_championnat_detectee_domicile'].fillna(False)
    & pairs['couverture_championnat_detectee_extérieur'].fillna(False)
    & pairs['elo_avant_match_domicile'].notna()
    & pairs['elo_avant_match_extérieur'].notna()
    & rest_difference.notna()
    & rest_difference.ne(0)
)
rested_home = rest_difference.gt(0)
rested_won = np.where(
    rested_home,
    pairs['resultat_domicile'].eq('Victoire'),
    pairs['resultat_extérieur'].eq('Victoire'),
)
rested_points = np.where(rested_home, pairs['points_reels_domicile'], pairs['points_reels_extérieur'])
rested_expected = np.where(rested_home, pairs['points_attendus_elo_domicile'], pairs['points_attendus_elo_extérieur'])
residual = rested_points - rested_expected
n = int(valid.sum())
missing_coverage = int((
    ~pairs['couverture_championnat_detectee_domicile'].fillna(False)
    | ~pairs['couverture_championnat_detectee_extérieur'].fillna(False)
).sum())
print(f'Matchs exclus car couverture championnat absente pour au moins un club : {missing_coverage:,}')
print(f'Matchs avec repos différent, couverture détectée et Elo des deux clubs : {n:,}')
if n:
    wins = int(rested_won[valid.to_numpy()].sum())
    mean_residual = float(np.mean(residual[valid.to_numpy()]))
    if n > 1:
        margin = 1.96 * float(np.std(residual[valid.to_numpy()], ddof=1)) / np.sqrt(n)
        ci_low, ci_high = mean_residual - margin, mean_residual + margin
        print(f'Le club le plus reposé gagne {wins}/{n} matchs ({wins / n:.1%}); cette statistique brute ne corrige pas la force.')
        print(f'Écart moyen aux points attendus par Elo : {mean_residual:+.3f} (IC 95 % descriptif : [{ci_low:+.3f}, {ci_high:+.3f}]).')
        if ci_low <= 0 <= ci_high:
            print('Conclusion prudente : cet échantillon ne montre pas de différence nette par rapport aux attentes Elo; cela ne prouve pas un effet exactement nul.')
        else:
            print('Conclusion prudente : l’écart aux attentes Elo ne contient pas zéro dans cet intervalle descriptif; vérifier avec un modèle et un test adaptés avant toute conclusion causale.')
    else:
        print(f'Écart aux points attendus par Elo : {mean_residual:+.3f}; un seul match ne permet pas d’estimer un intervalle.')
else:
    print('Pas assez de matchs admissibles pour comparer repos et attentes Elo.')

## Taux de victoire selon le repos

Pour chaque catégorie, le taux est le nombre de victoires divisé par toutes les observations club-match de la catégorie. Les nuls restent au dénominateur et ne sont pas des victoires.

In [ ]:
couverture_ok = fatigue['couverture_championnat_detectee']
fatigue_couverte = fatigue.loc[couverture_ok].copy()
print(f"Observations club-match exclues du graphique pour couverture championnat absente : {(~couverture_ok).sum():,}")

bins = [-np.inf, 3, 5, 8, np.inf]
labels = ['Moins de 3 jours', '3 à 4 jours', '5 à 7 jours', 'Plus de 7 jours']
fatigue_couverte['categorie_repos'] = pd.cut(
    fatigue_couverte['jours_repos'], bins=bins, labels=labels, right=False
)
rates = (
    fatigue_couverte.dropna(subset=['categorie_repos'])
    .groupby('categorie_repos', observed=False)
    .agg(matchs=('resultat', 'size'), victoires=('resultat', lambda values: values.eq('Victoire').sum()))
)
rates['taux_victoire_pct'] = rates['victoires'] / rates['matchs'] * 100
display(rates.round(1))

fig, ax = plt.subplots(figsize=(9, 4.5))
values = rates['taux_victoire_pct'].fillna(0)
bars = ax.bar(labels, values, color=[ORANGE, GRIS, BLEU, VERT], width=0.65)
for bar, (_, row) in zip(bars, rates.iterrows()):
    if row['matchs']:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f"{row['taux_victoire_pct']:.1f} %\n(n={int(row['matchs'])})",
            ha='center', va='bottom', fontsize=9,
        )
ax.set_ylabel('Victoires parmi les observations club-match (%)')
ax.set_xlabel('Jours de repos avant le match de C1')
ax.set_title('Taux de victoire selon le repos (calendrier championnat détecté)')
ax.set_ylim(0, max(100, values.max() + 12))
ax.spines[['top', 'right']].set_visible(False)
plt.xticks(rotation=10, ha='right')
plt.tight_layout()
plt.show()

## Limites et conclusion prudente

Le nombre de matchs ne mesure ni l'intensité, ni les prolongations, ni les compositions, ni les minutes disputées par les joueurs. L'absence de match `domestic_league` pour une saison signale un calendrier non couvert et ces observations sont exclues; la présence d'au moins un match ne garantit pas une couverture complète. L'Elo n'est disponible que pour les clubs reliés et les dates couvertes par `EloRatings.csv`; les résultats ajustés portent donc sur ce sous-échantillon. Enfin, un intervalle qui contient zéro signifie que cet échantillon ne met pas en évidence de différence nette aux attentes Elo, et non que l'effet réel est certainement nul. Corrélation, ajustement Elo et causalité sont des questions distinctes.